In [6]:
import rdflib
from rdflib.plugins.sparql import prepareQuery
from tabulate import tabulate

In [7]:
filename = "../data/14/ABox.ttl"

In [8]:
text1 = '''CQ_14.1
Return the characteristics of all agents that are people.
'''

query1 = '''
PREFIX aat: <http://vocab.getty.edu/aat/>
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>

SELECT ?full_name ?f_name ?l_name ?country ?language ?gender ?profession ?date_birth ?date_death (GROUP_CONCAT(DISTINCT ?link; separator="; ") AS ?links)
WHERE {
    ?agent a crm:E21_Person .
    OPTIONAL { ?agent crm:P1_is_identified_by [ a crm:E41_Appellation; crm:P2_has_type aat:300404688; crm:P190_has_symbolic_content ?full_name ] }
    OPTIONAL { ?agent crm:P1_is_identified_by [ a crm:E41_Appellation; crm:P2_has_type aat:300404651; crm:P190_has_symbolic_content ?f_name ] }
    OPTIONAL { ?agent crm:P1_is_identified_by [ a crm:E41_Appellation; crm:P2_has_type aat:300404652; crm:P190_has_symbolic_content ?l_name ] }
    OPTIONAL { ?agent crm:P74_has_current_or_former_residence/crm:P1_is_identified_by/crm:P190_has_symbolic_content ?country }
    OPTIONAL { ?agent crm:P67i_is_referred_to_by [ a crm:E33_Linguistic_Object; crm:P190_has_symbolic_content ?profession ] }
    OPTIONAL { 
    ?aa1 crm:P140_assigned_attribute_to ?agent ; crm:P177_assigned_property_of_type aat:300411835 ; crm:P141_assigned ?g_val .
    BIND(STR(?g_val) AS ?gender)
    }
    OPTIONAL { ?aa2 crm:P140_assigned_attribute_to ?agent ; crm:P177_assigned_property_of_type aat:300435433 ; crm:P141_assigned ?language }
    OPTIONAL { 
    ?agent crm:P12i_was_present_at [ a crm:E5_Event; crm:P2_has_type aat:300069672; crm:P4_has_time-span/crm:P82a_begin_of_the_begin ?b_dt ] 
    BIND(STRBEFORE(STR(?b_dt), "T") AS ?date_birth)
    }
    OPTIONAL { 
        ?agent crm:P12i_was_present_at [ a crm:E5_Event; crm:P2_has_type aat:300151836; crm:P4_has_time-span/crm:P82a_begin_of_the_begin ?d_dt ] 
        BIND(STRBEFORE(STR(?d_dt), "T") AS ?date_death)
    }
    OPTIONAL { ?agent crm:P1_is_identified_by [ a crm:E42_Identifier; crm:P2_has_type aat:300404629; crm:P190_has_symbolic_content ?link ] }
}
GROUP BY ?full_name ?f_name ?l_name ?country ?language ?gender ?profession ?date_birth ?date_death
'''

In [9]:
text2 = '''CQ_14.2
Return the characteristics of all agents that are organisations.
'''

query2 = '''
PREFIX aat: <http://vocab.getty.edu/aat/>
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>

SELECT ?name ?country ?s_date ?e_date (GROUP_CONCAT(DISTINCT ?link; separator="; ") AS ?links)
WHERE {
  ?org a crm:E74_Group .
  OPTIONAL { ?org crm:P1_is_identified_by [ a crm:E41_Appellation; crm:P2_has_type aat:300404688; crm:P190_has_symbolic_content ?name ] }
  OPTIONAL { ?org crm:P74_has_current_or_former_residence/crm:P1_is_identified_by/crm:P190_has_symbolic_content ?country }
  OPTIONAL { 
    ?org crm:P12i_was_present_at [ a crm:E5_Event; crm:P2_has_type aat:300393213; crm:P4_has_time-span/crm:P82a_begin_of_the_begin ?sd ] 
    BIND(STRBEFORE(STR(?sd), "T") AS ?s_date)
  }
  OPTIONAL { 
    ?org crm:P12i_was_present_at [ a crm:E5_Event; crm:P2_has_type aat:300393214; crm:P4_has_time-span/crm:P82a_begin_of_the_begin ?ed ] 
    BIND(STRBEFORE(STR(?ed), "T") AS ?e_date)
  }
  OPTIONAL { ?org crm:P1_is_identified_by [ a crm:E42_Identifier; crm:P2_has_type aat:300404629; crm:P190_has_symbolic_content ?link ] }
}
GROUP BY ?name ?country ?s_date ?e_date
'''

In [10]:
queries = [(text1, query1),
            (text2, query2),
           ]

g = rdflib.ConjunctiveGraph()
g.parse(filename, format="turtle", encoding="utf-8")

for query in queries:
    q = prepareQuery(query[1])
    results = g.query(q)
    print(query[0])
    table = []
    for row in results:
        table.append([row[var] for var in results.vars])
    print(tabulate(table, headers=results.vars, tablefmt="psql"))

CQ_14.1
Return the characteristics of all agents that are people.

+-------------+----------+----------+-------------------------------------------------+-------------------------------------------+--------------------------------------+-------------------------------------------------------------------------------------+--------------+--------------+----------------------------------------------------------------------------------------------------------+
| full_name   | f_name   | l_name   | country                                         | language                                  | gender                               | profession                                                                          | date_birth   | date_death   | links                                                                                                    |
|-------------+----------+----------+-------------------------------------------------+-------------------------------------------+--------------